# Dataset overview

This notebook inspects the cleaned silver transaction table: schema, row counts, class balance, and basic quality metrics.

**Prerequisites:** run `make sample-data` and `make ingest` from the repository root first.

In [1]:
from transaction_risk.spark.session import create_spark_session_from_yaml
spark = create_spark_session_from_yaml('../conf/spark.local.yaml')


In [2]:
from transaction_risk.spark.io import read_table

transactions = read_table(spark, '../data/silver/transactions')
transactions.printSchema()
print('rows:', transactions.count())
transactions.show(5)

root
 |-- transaction_id: string (nullable = true)
 |-- step: integer (nullable = true)
 |-- type: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- nameOrig: string (nullable = true)
 |-- oldbalanceOrg: double (nullable = true)
 |-- newbalanceOrig: double (nullable = true)
 |-- nameDest: string (nullable = true)
 |-- oldbalanceDest: double (nullable = true)
 |-- newbalanceDest: double (nullable = true)
 |-- isFraud: integer (nullable = true)
 |-- isFlaggedFraud: integer (nullable = true)



rows: 5000


+--------------------+----+--------+-------+--------+-------------+--------------+--------+--------------+--------------+-------+--------------+
|      transaction_id|step|    type| amount|nameOrig|oldbalanceOrg|newbalanceOrig|nameDest|oldbalanceDest|newbalanceDest|isFraud|isFlaggedFraud|
+--------------------+----+--------+-------+--------+-------------+--------------+--------+--------------+--------------+-------+--------------+
|00003f3525d6a9626...| 168|CASH_OUT|2703.83|C0001028|    147912.76|     145208.93|C0001163|      28122.43|      30826.26|      0|             0|
|0032de6f7e8c26615...| 142|CASH_OUT|1545.55|C0000271|     13920.67|      12375.12|C0000410|     104993.37|     106538.92|      0|             0|
|0033d80b86431e602...| 207|CASH_OUT| 7418.3|C0000024|    144715.97|     137297.67|C0001003|     263842.43|     271260.73|      0|             0|
|0051ad18b07ceb509...| 104|TRANSFER|3203.59|C0001075|      59575.3|      56371.71|C0000200|      16027.26|      19230.85|      0| 

In [3]:
from transaction_risk.validation.data_quality import basic_quality_report, class_balance_report

print(basic_quality_report(transactions))
class_balance_report(transactions).show()

{'row_count': 5000, 'column_count': 12, 'fraud_count': 82, 'fraud_rate': 0.0164}


+-------+-----+----------+
|isFraud|count|proportion|
+-------+-----+----------+
|      0| 4918|    0.9836|
|      1|   82|    0.0164|
+-------+-----+----------+



In [4]:
from transaction_risk.validation.expectations import (
    expectation_suite_to_markdown,
    run_transaction_expectation_suite,
)

suite = run_transaction_expectation_suite(transactions, dataset_name='silver_transactions')
print(expectation_suite_to_markdown(suite))
spark.stop()

# Data quality report — silver_transactions

Overall status: **PASSED**

| check | status | details |
| --- | --- | --- |
| columns_exist | pass | required=step, type, amount, isFraud |
| completeness_amount | pass | column=amount; non_null_ratio=1.0000; min_ratio=0.9900 |
| completeness_type | pass | column=type; non_null_ratio=1.0000; min_ratio=0.9900 |
| completeness_step | pass | column=step; non_null_ratio=1.0000; min_ratio=0.9900 |
| binary_label_isFraud | pass | column=isFraud; invalid_count=0 |
| non_negative_amount | pass | column=amount; negative_count=0 |
| non_negative_step | pass | column=step; negative_count=0 |
| duplicate_rate | pass | key_columns=transaction_id; duplicate_ratio=0.0000; max_ratio=0.0100 |

